# Applying Facial Recognition to a New Dataset: Newspaper Photos

This is the "apply the technique to something new" part of my Advanced DH Technique assignment — applying the **facial recognition** pipeline from my [first notebook](../facial-recognition-yearbooks/Wolf-FacialRecognition-Notebook.html) (Haar Cascade face detection + DeepFace emotion classification) to a dataset I haven't run it on before.

**Dataset pivot:** I originally tried downloading Denison University's own yearbook (*The Adytum*) from the Internet Archive for this step, but three different years' PDFs stalled or silently failed mid-download in Colab. Rather than keep fighting an unreliable source, I switched to a dataset I already know downloads fast and reliably: the **Newspaper Navigator photo dataset** hosted on Zenodo, the same source I used in my Computer Vision for the Humanities notebook. It's a genuinely different, never-before-analyzed-for-faces dataset — in that notebook I only classified these photos by subject matter (human/animal/landscape/human-structure); I never ran face or emotion detection on them.

**Dataset:** [multi_label.csv](https://zenodo.org/records/4487141/files/multi_label.csv?download=1) + [images.zip](https://zenodo.org/records/4487141/files/images.zip?download=1) — newspaper photos from the Library of Congress's *Chronicling America* collection, 1895-1920, via the Newspaper Navigator project. I filter down to just the photos already labeled `"human"` (about 1,600 of them) and sample a subset for a reasonable runtime.

**Run this in Google Colab** (no GPU needed).

In [ ]:
from google.colab import _message
import json

nb = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']
with open('Wolf-FacialRecognitionNewDataset-Notebook.ipynb', 'w') as f:
    json.dump(nb, f)

!jupyter nbconvert --to html Wolf-FacialRecognitionNewDataset-Notebook.ipynb

from google.colab import files
files.download('Wolf-FacialRecognitionNewDataset-Notebook.ipynb')
files.download('Wolf-FacialRecognitionNewDataset-Notebook.html')

## Step 1: Setup

Downloads the same Zenodo-hosted photo dataset from my second notebook. No PDF-to-image conversion needed this time, since these are already individual `.jpg` files — that removes an entire, more fragile step from the original pipeline.

In [1]:
import urllib.request, os

def download(url, dest, min_bytes=1_000_000, retries=3):
    for attempt in range(1, retries + 1):
        if os.path.exists(dest) and os.path.getsize(dest) >= min_bytes:
            print(f"{dest}: already downloaded ({os.path.getsize(dest):,} bytes)")
            return
        if os.path.exists(dest):
            os.remove(dest)
        print(f"Downloading {dest} (attempt {attempt}/{retries})...")
        try:
            urllib.request.urlretrieve(url, dest)
            size = os.path.getsize(dest)
            print(f"  got {size:,} bytes")
            if size >= min_bytes:
                return
            print("  file too small, retrying")
        except Exception as e:
            print(f"  failed: {e}")
    raise RuntimeError(f"Failed to download {dest} after {retries} attempts")

download("https://zenodo.org/records/4487141/files/multi_label.csv?download=1", "multi_label.csv", min_bytes=100_000)
download("https://zenodo.org/records/4487141/files/images.zip?download=1", "photo_images.zip", min_bytes=800_000_000)

!unzip -n -q photo_images.zip -d images/
!echo "images found:" && find images -type f | wc -l

!pip install -q dlib
!pip install -q DeepFace

import shutil, cv2, numpy as np, pandas as pd, dlib, tensorflow as tf
from deepface import DeepFace

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"


  got 168,478 bytes
  got 880,141,662 bytes
images found:
2003
26-11-09 22:27:43 - Directory /root/.deepface has been created
26-11-09 22:27:43 - Directory /root/.deepface/weights has been created


## Step 2: Filtering to Human-Labeled Photos

Running face detection on landscape/animal/building photos would just waste time, so I filter down to photos already labeled `"human"` (using the same label-splitting logic from my second notebook), then sample a subset (300 photos) to keep the runtime reasonable — DeepFace's emotion analysis takes a few seconds per face.

In [2]:
df = pd.read_csv('multi_label.csv', na_filter=False)

df['label_list'] = df['label'].apply(lambda s: s.split('|'))
human_df = df[df['label_list'].apply(lambda labels: 'human' in labels)]

print(f"{len(human_df)} photos labeled 'human' out of {len(df)} total")

sample_df = human_df.sample(n=min(300, len(human_df)), random_state=42)
print(f"Sampling {len(sample_df)} photos for face detection")

os.makedirs('sample_images', exist_ok=True)
for fname in sample_df['download_image_path']:
    src = os.path.join('images', fname)
    dst = os.path.join('sample_images', fname)
    if os.path.exists(src):
        shutil.copy(src, dst)

print("images copied:", len(os.listdir('sample_images')))


1598 photos labeled 'human' out of 2000 total
Sampling 300 photos for face detection
images copied: 300


## Step 3: Face Detection

Same Haar Cascade approach as my first notebook — but simpler, since there's no per-year folder structure to manage here, just one flat set of photos.

In [3]:
os.makedirs('faces', exist_ok=True)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

count = 0
for fname in os.listdir('sample_images'):
    image = cv2.imread(os.path.join('sample_images', fname))
    if image is None:
        continue

    greyscale_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    detected_faces = face_cascade.detectMultiScale(image=greyscale_image, scaleFactor=1.9, minNeighbors=4)

    for (x, y, w, h) in detected_faces:
        try:
            xpadding = 20
            ypadding = 40
            crop_face = image[y-ypadding: y + h+ypadding, x-xpadding: x + w+xpadding]
            count += 1
            face = cv2.rectangle(crop_face, (x, y), (x+w, y+h), (255, 0, 0), 2)
            cv2.imwrite(f'faces/{count}_{fname}', face)
        except (Exception):
            print("An error happened")
            continue

print(f"{count} faces detected and cropped")


210 faces detected and cropped


## Step 4: Smile Identification

Also simpler than my first notebook's version — no year-boundary bookkeeping needed here, since there's only one group of photos, not six years' worth. Just loop through every detected face, run DeepFace, and tally the results.

In [4]:
number_smiles = 0
number_nonsmiles = 0
num_errors = 0

for fname in os.listdir('faces'):
    try:
        demography = DeepFace.analyze(os.path.join('faces', fname), actions=['emotion'])
        if demography[0]['dominant_emotion'] == 'happy':
            number_smiles += 1
        else:
            number_nonsmiles += 1
    except (Exception):
        num_errors += 1
        continue

total = number_smiles + number_nonsmiles + num_errors
print(f"Smiles: {number_smiles} ({number_smiles/total:.1%})")
print(f"Non-smiles: {number_nonsmiles} ({number_nonsmiles/total:.1%})")
print(f"Errors: {num_errors} ({num_errors/total:.1%})")

data = pd.DataFrame([{
    'Smiles': number_smiles / total,
    'Non-Smiles': number_nonsmiles / total,
    'Error Weight': num_errors / total,
}])
data.to_csv('NewspaperPhotosOutput.csv', index=False)
data


26-11-09 22:30:42 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 72.1MB/s]


Smiles: 20 (9.5%)
Non-smiles: 173 (82.4%)
Errors: 17 (8.1%)


,Smiles,Non-Smiles,Error Weight
0,0.095238,0.82381,0.080952


## Step 5: Download Results

In [5]:
from google.colab import files
files.download('NewspaperPhotosOutput.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reflection

**Reflection question 1 (on the pivot itself):** *My original plan here was Denison's own yearbook, The Adytum, pulled from the Internet Archive. Three different years' PDFs stalled or silently failed mid-download in Colab, so I switched to this Zenodo-hosted dataset instead. What does needing to make that kind of switch mid-project say about relying on third-party-hosted historical data for a time-sensitive assignment?*

> Your answer here:
>
>

**Reflection question 2:** *Your first notebook ran facial recognition on posed, formal yearbook portraits spanning six decades. This dataset is newspaper photojournalism from a single roughly 25-year window (1895-1920) — candid news photos, not posed portraits, with no year-by-year structure to compare smile trends against. What's one way the *type* of historical photo (posed portrait vs. candid news photo) might affect smile-detection results, independent of the time period?*

> Your answer here:
>
>
